In [6]:
import pandas as pd
import os

# Pfade zentral definieren (macht das Projekt robust)
RAW_DATA = "../data/raw/WDIData.csv"
PROCESSED_DATA = "../data/processed/wdi_cleaned.csv"

print("Libraries erfolgreich geladen.")

Libraries erfolgreich geladen.


In [7]:
df_raw = pd.read_csv(RAW_DATA)

# Kurze Inspektion für die Doku
print(f"Der Datensatz hat {df_raw.shape[0]} Zeilen und {df_raw.shape[1]} Spalten.")
df_raw.head(3)

Der Datensatz hat 422136 Zeilen und 64 Spalten.


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,Unnamed: 63
0,Arab World,ARB,"2005 PPP conversion factor, GDP (LCU per inter...",PA.NUS.PPP.05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Arab World,ARB,"2005 PPP conversion factor, private consumptio...",PA.NUS.PRVT.PP.05,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Arab World,ARB,Access to clean fuels and technologies for coo...,EG.CFT.ACCS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,82.407647,82.827636,83.169227,83.587141,83.954293,84.23063,84.570425,NaN,NaN,NaN


In [8]:
# Definition der Ziel-Variablen
indicators = {
    'IT.NET.USER.ZS': 'internet_users',
    'EN.ATM.CO2E.PC': 'co2_per_capita',
    'NY.GDP.PCAP.CD': 'gdp_per_capita'
}

# Filtern auf relevante Zeilen
df_filtered = df_raw[df_raw['Indicator Code'].isin(indicators.keys())].copy()
df_filtered['indicator'] = df_filtered['Indicator Code'].map(indicators)

print(f"Gefiltert auf {len(indicators)} Indikatoren.")

Gefiltert auf 3 Indikatoren.


In [9]:
# Jahre identifizieren (alle Spalten, die nur aus Zahlen bestehen)
id_vars = ['Country Name', 'Country Code', 'indicator']
year_columns = [col for col in df_raw.columns if col.isdigit()]

# Von Wide zu Long Format
df_long = df_filtered.melt(id_vars=id_vars, value_vars=year_columns, 
                           var_name='Year', value_name='Value')

# Pivotieren: Jeder Indikator bekommt eine eigene Spalte
df_tidy = df_long.pivot_table(index=['Country Name', 'Country Code', 'Year'], 
                               columns='indicator', 
                               values='Value').reset_index()

df_tidy['Year'] = df_tidy['Year'].astype(int)
df_tidy.head()

indicator,Country Name,Country Code,Year,co2_per_capita,gdp_per_capita,internet_users
0,Afghanistan,AFG,1960,0.046060,59.777327,NaN
1,Afghanistan,AFG,1961,0.053604,59.878153,NaN
2,Afghanistan,AFG,1962,0.073765,58.492874,NaN
3,Afghanistan,AFG,1963,0.074233,78.782758,NaN
4,Afghanistan,AFG,1964,0.086292,82.208444,NaN


In [10]:
# Fokus auf das digitale Zeitalter
df_clean = df_tidy[df_tidy['Year'] >= 2000].copy()

# Statistik der fehlenden Werte vor dem Drop (gut für die Kritik am Datensatz)
print("Fehlende Werte vor Bereinigung:")
print(df_clean.isnull().sum())

# Zeilen ohne Kern-Daten entfernen
df_clean = df_clean.dropna(subset=['internet_users', 'co2_per_capita'])

print(f"Verbleibende Datenpunkte nach Bereinigung: {len(df_clean)}")

Fehlende Werte vor Bereinigung:
indicator
Country Name        0
Country Code        0
Year                0
co2_per_capita    929
gdp_per_capita    180
internet_users    210
dtype: int64
Verbleibende Datenpunkte nach Bereinigung: 3608


In [11]:
# Ordner erstellen, falls nicht vorhanden
os.makedirs("../data/processed", exist_ok=True)

# Speichern
df_clean.to_csv(PROCESSED_DATA, index=False)
print(f"Datei erfolgreich gespeichert: {PROCESSED_DATA}")

Datei erfolgreich gespeichert: ../data/processed/wdi_cleaned.csv
